# 재정렬(리랭커) 비교 실험

같은 검색 후보를 두고 **재정렬 방식만 바꿔서** 정확도와 속도를 비교해요.

| 구성 | 재정렬 | 모델 |
| --- | --- | --- |
| R0 | 없음 (Dense + BM25 → RRF) | |
| R1 | ColBERT | BGE-M3 (검색과 같은 모델) |
| R2 | 크로스 인코더 | bge-reranker-v2-m3 |

- 문항: 골든셋. 설정은 **dev로만** 정하고, test는 마지막에 한 번만 돌려요.
- 채점: 조항 단위. 찾은 조각이 정답 조각과 같거나 같은 문서·같은 조항이면 맞음.
- LLM은 쓰지 않아요. 검색 순위만 재요.

**준비**: `rerank_exp` 폴더를 통째로 구글 드라이브(내 드라이브)에 올려요. 폴더 안에 이 노트북과 `rerank_exp_pack.zip`이 같이 있어요.
**실행**: 런타임 → 런타임 유형 변경 → GPU(T4 또는 L4) → 모두 실행

실행하는 동안 셀 아래에 진행 상황이 시각과 함께 찍혀요. 같은 내용이 폴더의 `run_log_<RUN_TAG>.txt`에도 저장돼요.
결과(`rerank_results_<RUN_TAG>`)와 임베딩 저장분(`rag_cache`)도 같은 폴더에 생겨요.

## 1. 설정 (여기만 고치면 돼요)

In [ ]:
DATA_DIR = "/content/drive/MyDrive/rerank_exp"   # 이 노트북과 rerank_exp_pack.zip이 있는 폴더. 결과도 여기에 저장
PACK = "rerank_exp_pack.zip"
USE_DRIVE = True
RUN_TAG = "v1"                    # 결과 폴더 이름: rerank_results_<RUN_TAG>

TUNE_SPLITS = ["dev"]             # 설정을 정할 때 쓰는 묶음
FINAL_SPLITS = ["test", "불확실", "법령근거없음"]   # 설정을 확정한 뒤 한 번만 (불확실·법령 문항도 정답 조문은 있어요)
RUN_FINAL = False                 # dev로 설정을 정한 뒤 True로 바꿔 한 번만 실행

BASE_PRESET = "A1"                # 재정렬 전 검색: A1 = Dense + BM25, 카테고리 필터 없음 (재정렬 효과만 보려고)
DOC_KINDS = []                    # [] = 규정 + 법령 전부, ["내부규정"] = 법령 빼고
RERANK_POOL = 20                  # 재정렬할 후보 수
TOP_K = 8                         # LLM에 넘기는 조각 수

CONFIGS = {
    "R0 재정렬 없음": {"rerank": "none"},
    "R1 ColBERT": {"rerank": "colbert"},
    "R2 크로스 인코더": {"rerank": "cross"},
}
EXTRA_CONFIGS = {                 # dev에서 더 비교해 보고 싶은 설정 (필요 없으면 비워 두세요)
    # "R1 ColBERT + Dense 반반": {"rerank": "colbert", "rerank_dense_weight": 0.5},
    # "R2 후보 10개": {"rerank": "cross", "rerank_pool": 10},
}
LOG_EVERY = 10                    # 문항 몇 개마다 진행 상황을 찍을지
MOCK = False                      # True면 모델 없이 가짜로 코드만 점검 (점수는 의미 없음)

## 2. 환경 준비

In [ ]:
import os, sys, time, json, zipfile, shutil, glob, logging
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !nvidia-smi --query-gpu=name,memory.total --format=csv
    print("패키지 설치 중이에요 (2~3분). 끝나면 '설치 완료'가 찍혀요.", flush=True)
    t_inst = time.time()
    if not MOCK:
        !pip -q install -U FlagEmbedding sentence-transformers kiwipiepy
    else:
        !pip -q install kiwipiepy
    print(f"설치 완료 ({time.time() - t_inst:.0f}초)", flush=True)

In [ ]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# 폴더 찾기: DATA_DIR에 묶음 파일이 없으면 드라이브에서 찾아 봐요 (폴더 이름을 바꿨거나 다른 곳에 올린 경우)
if not os.path.exists(os.path.join(DATA_DIR, PACK)) and IN_COLAB:
    found = sorted(glob.glob(f"/content/drive/MyDrive/**/{PACK}", recursive=True))
    if found:
        DATA_DIR = os.path.dirname(found[0])
        print("묶음 파일을 여기서 찾았어요:", DATA_DIR)
pack_path = os.path.join(DATA_DIR, PACK)
assert os.path.exists(pack_path), f"{pack_path} 가 없어요. rerank_exp 폴더를 내 드라이브에 올렸는지 확인해 주세요."

WORK = "/content/rerank_exp_work" if IN_COLAB else os.path.join(DATA_DIR, "_work")
OUT_DIR = os.path.join(DATA_DIR, f"rerank_results_{RUN_TAG}")
CACHE_DIR = os.path.join(DATA_DIR, "rag_cache")        # 조각 임베딩을 저장해 두는 곳 (다음 실행부터 건너뜀)
LOG_PATH = os.path.join(DATA_DIR, f"run_log_{RUN_TAG}.txt")
os.makedirs(OUT_DIR, exist_ok=True)

# 진행 상황: 셀 아래와 run_log 파일에 같이 남겨요 (시각은 한국 시간)
class KST(logging.Formatter):
    converter = staticmethod(lambda t: time.gmtime(t + 9 * 3600))
fmt = KST("%(asctime)s  %(message)s", "%H:%M:%S")
handlers = [logging.StreamHandler(sys.stdout), logging.FileHandler(LOG_PATH, encoding="utf-8")]
for h in handlers:
    h.setFormatter(fmt)
root = logging.getLogger("rag")
root.handlers, root.propagate = handlers, False
root.setLevel(logging.INFO)
log = root.info

def step(msg):
    log("=" * 8 + " " + msg + " " + "=" * 8)

step(f"실행 시작 (RUN_TAG={RUN_TAG}, MOCK={MOCK})")
shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(pack_path).extractall(WORK)
log(f"폴더: {DATA_DIR}")
log(f"결과: {OUT_DIR}")
log(f"로그: {LOG_PATH}")

# 묶음 안에서 코드(rag_pkg)와 데이터(data) 위치를 찾아요. 압축을 다시 묶어서 폴더가 한 겹 더 있어도 찾아요.
def tree(d, depth=2):
    out = []
    for root, dirs, files in os.walk(d):
        if root[len(d):].count(os.sep) < depth:
            out += [os.path.relpath(os.path.join(root, x), d) for x in sorted(dirs + files)[:12]]
    return out[:40]

hits = sorted(glob.glob(os.path.join(WORK, "**", "rag_pkg", "rag", "__init__.py"), recursive=True))
assert hits, (f"묶음 파일({pack_path}, {os.path.getsize(pack_path):,}바이트) 안에서 rag_pkg를 찾지 못했어요. "
              f"들어 있는 것: {tree(WORK)}. PC의 rerank_exp 폴더에 있는 rerank_exp_pack.zip을 그대로 다시 올려 주세요.")
PACK_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
PKG, DATA = os.path.join(PACK_ROOT, "rag_pkg"), os.path.join(PACK_ROOT, "data")
assert os.path.exists(os.path.join(DATA, "golden_set.json")), f"묶음 안에 data 폴더가 없어요. 들어 있는 것: {tree(WORK)}"
sys.path.insert(0, PKG)
log(f"묶음 위치: {PACK_ROOT} | 내용: {sorted(os.listdir(PACK_ROOT))} | data: {sorted(os.listdir(DATA))}")

## 3. 색인 만들기
조각을 읽고, BGE-M3로 임베딩하고(처음 한 번, 1~3분), BM25 색인을 만들어요.

In [ ]:
import os, sys, zipfile, importlib
# 앞 셀이 끝까지 실행됐는지 확인해요 (세션을 다시 시작했거나 앞 셀에서 오류가 났으면 여기서 알려 줘요)
assert all(k in globals() for k in ("WORK", "pack_path", "log", "step", "PKG", "DATA")), (
    "바로 위 '2. 환경 준비'의 두 번째 셀이 끝까지 실행되지 않았어요. "
    "그 셀의 출력에 오류가 있는지 확인하고, '런타임 > 모두 실행'으로 처음부터 다시 실행해 주세요. "
    "(설치 뒤에 '세션 다시 시작'을 눌렀다면 설정 셀부터 다시 실행해야 해요.)")
if PKG not in sys.path:
    sys.path.insert(0, PKG)
importlib.invalidate_caches()

import numpy as np, pandas as pd
from rag import Pipeline, load_config
from rag.embedder import HashEmbedder
from rag.goldenset import evaluate, load_items

cfg = load_config().with_(
    hrdb_url="", chunks=(os.path.join(DATA, "chunks_regs.jsonl"), os.path.join(DATA, "chunks_laws.jsonl")),
    doc_kinds=tuple(DOC_KINDS), cache_dir=CACHE_DIR, rerank_pool=RERANK_POOL, n_final=max(20, RERANK_POOL), top_k=TOP_K,
    th_answer=0.0,                 # 순위만 재려고 "근거 없음" 판단은 꺼요
).preset(BASE_PRESET)

step("색인 만들기")
t0 = time.time()
pipe = Pipeline(cfg, embedder=HashEmbedder() if MOCK else "auto")
log(json.dumps(pipe.info(), ensure_ascii=False) + f" | {time.time() - t0:.0f}초")
assert pipe.retriever.dense_ready, "Dense 검색이 꺼져 있어요. 위 설치 셀의 오류를 확인해 주세요."
if MOCK:
    print("⚠️ MOCK 모드예요. 코드 점검용이고 점수는 의미가 없어요.")
if "colbert" in [c.get("rerank") for c in {**CONFIGS, **EXTRA_CONFIGS}.values()]:
    assert hasattr(pipe.embedder, "colbert"), "ColBERT는 FlagEmbedding이 필요해요. 설치 셀을 확인해 주세요."

ITEMS = load_items(os.path.join(DATA, "golden_set.json"))
from collections import Counter
log(f"골든셋 {len(ITEMS)}문항 | {dict(Counter(i['split'] for i in ITEMS))}")

## 4. 질문 몇 개로 눈으로 확인

In [ ]:
class MockCross:                   # MOCK 모드에서만 쓰는 가짜 크로스 인코더
    name = "mock-cross"
    def score(self, query, docs):
        return [len(set(query) & set(d[:200])) / 50 for d in docs]

if MOCK:
    pipe.retriever.set_cross(cfg.rerank_model, MockCross())

def title(cid):
    r = pipe.retriever.rows[cid]
    return f'{r["doc_name"]} {r["article"]}'.strip()

ALL_CONFIGS = {**CONFIGS, **EXTRA_CONFIGS}
step("질문 3개로 확인 (재정렬 모델을 처음 불러와서 조금 걸려요)")
for q in ["결혼하면 휴가 며칠 받아요?", "서울 출장 가면 숙박비 얼마까지 나와?", "병가 쓸 때 진단서 필요해?"]:
    print("Q.", q)
    for name, opt in CONFIGS.items():
        r = pipe.with_config(cfg.with_(**opt)).run(q, force_search=True)
        print(f"   {name:14s}", " | ".join(title(c.chunk_id) for c in r.chunks[:3]))

## 5. dev 문항으로 비교

시간은 공정하게 재려고 구성마다 질문 쪽 저장분을 비우고 재요.
- **R1 ColBERT (처음)**: 조각의 토큰 벡터를 질문이 올 때 계산
- **R1 ColBERT (미리 계산)**: 조각의 토큰 벡터가 준비된 상태. 실제 서비스에서는 이렇게 미리 계산해 둬요

In [ ]:
try:
    import torch
    CUDA = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA else "CPU"
except Exception:
    CUDA, GPU_NAME = False, "CPU"

def gpu_peak_reset():
    if CUDA:
        torch.cuda.reset_peak_memory_stats()

def gpu_peak_gb():
    return round(torch.cuda.max_memory_allocated() / 1e9, 2) if CUDA else None

def run_config(name, opt, items, keep_docs=True, label=None):
    log(f"▶ {label or name} 시작 ({len(items)}문항)")
    p = pipe.with_config(cfg.with_(**opt))
    pipe.reset_query_cache(keep_docs=keep_docs)
    p.run("연차휴가는 며칠인가요?", force_search=True)          # 모델을 처음 불러오는 시간은 빼요
    pipe.reset_query_cache(keep_docs=keep_docs)
    gpu_peak_reset()
    t0 = time.time()
    out = evaluate(p, items, ks=(1, 3, 5, TOP_K), label=label or name, every=LOG_EVERY)
    s = out["summary"]
    log(f"■ {label or name} 끝: Hit@1 {s['hit@1']:.3f} · Hit@3 {s['hit@3']:.3f} · Hit@{TOP_K} {s[f'hit@{TOP_K}']:.3f} · "
        f"MRR {s['mrr']:.3f} · 문항당 {s['ms_mean']:.0f}ms · {time.time() - t0:.0f}초")
    row = {"구성": label or name, "문항": s["scored"], "Hit@1": s["hit@1"], "Hit@3": s["hit@3"], "Hit@5": s["hit@5"],
           f"Hit@{TOP_K}": s[f"hit@{TOP_K}"], "MRR": s["mrr"], "후보 안에 정답": s["in_pool"],
           "문항당 ms(평균)": s["ms_mean"], "문항당 ms(p95)": s["ms_p95"], "재정렬 ms(평균)": s["rerank_ms_mean"],
           "GPU 최대(GB)": gpu_peak_gb(), "전체(초)": round(time.time() - t0, 1)}
    for r in out["rows"]:
        r["config"] = label or name
    return row, out["rows"]

def run_all(items, configs):
    table, rows = [], {}
    for name, opt in configs.items():
        if opt.get("rerank") == "colbert":
            row, _ = run_config(name, opt, items, keep_docs=False, label=name + " (처음)")
            table.append(row)
            row, rr = run_config(name, opt, items, keep_docs=True, label=name + " (미리 계산)")
        else:
            row, rr = run_config(name, opt, items)
        table.append(row)
        rows[name] = rr
    return pd.DataFrame(table).set_index("구성"), rows

TUNE = [i for i in ITEMS if i["split"] in TUNE_SPLITS]
step(f"dev 비교: {len(TUNE)}문항 | GPU: {GPU_NAME} | 검색: {BASE_PRESET} | 후보 {RERANK_POOL}개 | 구성 {len(ALL_CONFIGS)}개")
tune_table, tune_rows = run_all(TUNE, ALL_CONFIGS)
tune_table

## 6. 재정렬이 고칠 수 있는 문항은 얼마나 되나

재정렬은 후보 안에서 순서만 바꿔요. 정답이 후보에 없으면 고칠 수 없어요.
아래 표에서 **4~20등** 문항이 재정렬로 3등 안에 올릴 수 있는 대상이에요.

In [ ]:
BASE = list(CONFIGS)[0]
def bucket(r0):
    if r0 is None: return "후보에 없음"
    if r0 == 1: return "1등"
    if r0 <= 3: return "2~3등"
    if r0 <= TOP_K: return f"4~{TOP_K}등"
    return f"{TOP_K + 1}~{RERANK_POOL}등"

base_df = pd.DataFrame(tune_rows[BASE])
base_df["재정렬 전 정답 위치"] = base_df["rank0"].apply(lambda x: bucket(None if pd.isna(x) else int(x)))
order = ["1등", "2~3등", f"4~{TOP_K}등", f"{TOP_K + 1}~{RERANK_POOL}등", "후보에 없음"]
head = base_df["재정렬 전 정답 위치"].value_counts().reindex(order, fill_value=0).to_frame("문항 수")
head["비율(%)"] = (head["문항 수"] / len(base_df) * 100).round(1)
head

## 7. 묶음별 비교
전체 점수에서는 안 보이는 차이를 봐요.

In [ ]:
rows_of = pipe.retriever.rows
def gold_kind(gold):
    arts = [rows_of[g]["article"] for g in gold if g in rows_of]
    return {"별표": any(a.startswith("별표") for a in arts), "법령": any(g.startswith("L") for g in gold)}

def subsets(df):
    g = df["gold"].apply(gold_kind)
    fixable = set(base_df[base_df["rank0"].between(4, RERANK_POOL)]["id"])
    return {
        "전체": df.index == df.index,
        "재정렬로 고칠 수 있는 문항 (전 4~20등)": df["id"].isin(fixable),
        "여러 조문이 필요한 문항 (multi_hop)": df["multi_hop"] == True,
        "정답이 별표(표)": g.apply(lambda x: x["별표"]),
        "정답이 법령": g.apply(lambda x: x["법령"]),
        "포괄 질문": df["type"] == "포괄 질문",
    }

def subset_table(rows_by_cfg):
    dfs = {name: pd.DataFrame(rr) for name, rr in rows_by_cfg.items()}
    out = []
    for label in subsets(next(iter(dfs.values()))):            # 묶음별로 구성을 나란히
        for name, df in dfs.items():
            d = df[subsets(df)[label]]
            if len(d):
                out.append({"묶음": label, "구성": name, "문항": len(d), "Hit@1": round(d["hit@1"].mean(), 3),
                            "Hit@3": round(d["hit@3"].mean(), 3), f"Hit@{TOP_K}": round(d[f"hit@{TOP_K}"].mean(), 3),
                            "MRR": round(d["rr"].mean(), 3)})
    return pd.DataFrame(out).set_index(["묶음", "구성"])

subset_table(tune_rows)

In [ ]:
# 카테고리별 Hit@3
cat = pd.concat([pd.DataFrame(rr).groupby("category")["hit@3"].mean().round(3).rename(name) for name, rr in tune_rows.items()], axis=1)
cat["문항"] = pd.DataFrame(tune_rows[BASE]).groupby("category").size()
cat

## 8. 재정렬로 순위가 바뀐 문항
좋아진 문항과 나빠진 문항을 같이 봐요. 나빠진 문항이 많으면 재정렬이 손해일 수 있어요.

In [ ]:
def rank_or(x, miss=99):
    return miss if pd.isna(x) else int(x)

def changes(rows_by_cfg, base=BASE):
    b = pd.DataFrame(rows_by_cfg[base]).set_index("id")
    summary, detail = [], {}
    for name, rr in rows_by_cfg.items():
        if name == base:
            continue
        d = pd.DataFrame(rr).set_index("id")
        j = b[["question", "rank"]].rename(columns={"rank": "전"}).join(d["rank"].rename("후"))
        j["전"], j["후"] = j["전"].apply(rank_or), j["후"].apply(rank_or)
        up, down = j[j["후"] < j["전"]], j[j["후"] > j["전"]]
        summary.append({"구성": name, "좋아짐": len(up), "나빠짐": len(down), "그대로": len(j) - len(up) - len(down),
                        "3등 밖 → 3등 안": int(((j["전"] > 3) & (j["후"] <= 3)).sum()),
                        "3등 안 → 3등 밖": int(((j["전"] <= 3) & (j["후"] > 3)).sum())})
        detail[name] = j.assign(변화=j["전"] - j["후"]).sort_values("변화")
    return pd.DataFrame(summary).set_index("구성"), detail

chg_table, chg_detail = changes(tune_rows)
display(chg_table)
for name, d in chg_detail.items():
    print(f"\n[{name}] 가장 나빠진 문항 (99 = {TOP_K}등 밖)")
    display(d.head(5)[["question", "전", "후"]])
    print(f"[{name}] 가장 좋아진 문항")
    display(d.tail(5)[["question", "전", "후"]])

## 9. 어떤 구성으로도 못 찾은 문항
"규정은 있는데 못 찾는 질문"의 실제 목록이에요. 검색(질의 재작성, 사전, 조각 나누기)을 고칠 때와 골든셋을 보강할 때 써요.

In [ ]:
def never_found(rows_by_cfg):
    ok = set()
    for rr in rows_by_cfg.values():
        ok |= {r["id"] for r in rr if r["rank"] is not None}
    b = pd.DataFrame(rows_by_cfg[BASE])
    nf = b[~b["id"].isin(ok)][["id", "category", "type", "question", "gold", "top", "in_pool"]].copy()
    nf["정답 조문"] = nf["gold"].apply(lambda g: " / ".join(title(x) for x in g))
    nf["1등으로 나온 조문"] = nf["top"].apply(lambda t: title(t[0]) if t else "")
    return nf.drop(columns=["gold", "top"])

nf = never_found(tune_rows)
print(f"{TOP_K}등 안에 한 번도 못 든 문항: {len(nf)} / {len(tune_rows[BASE])}  (후보 20개 안에는 있었던 문항 {int(nf['in_pool'].sum())}개)")
nf.head(20)

## 10. 최종 실행 (test)

dev 결과로 설정(구성, 후보 수, 섞는 비율)을 정한 뒤, **1번 설정에서 `RUN_FINAL = True`로 바꾸고 한 번만** 실행해요.
test 결과를 보고 설정을 다시 바꾸면 점수가 부풀어요.

In [ ]:
final_table, final_rows = None, {}
if RUN_FINAL:
    FINAL = [i for i in ITEMS if i["split"] in FINAL_SPLITS]
    step(f"최종 실행: {len(FINAL)}문항 {FINAL_SPLITS}")
    final_table, final_rows = run_all(FINAL, CONFIGS)
    display(final_table)
    by_split = []
    for name, rr in final_rows.items():
        d = pd.DataFrame(rr)
        for sp, g in d.groupby("split"):
            by_split.append({"구성": name, "묶음": sp, "문항": len(g), "Hit@1": round(g["hit@1"].mean(), 3),
                             "Hit@3": round(g["hit@3"].mean(), 3), f"Hit@{TOP_K}": round(g[f"hit@{TOP_K}"].mean(), 3)})
    display(pd.DataFrame(by_split).set_index(["묶음", "구성"]))
else:
    log("RUN_FINAL = False 라서 최종 실행은 건너뛰었어요.")

## 11. 결과 저장

In [ ]:
meta = {"run_tag": RUN_TAG, "gpu": GPU_NAME, "mock": MOCK, "base_preset": BASE_PRESET, "doc_kinds": DOC_KINDS,
        "rerank_pool": RERANK_POOL, "top_k": TOP_K, "chunks": len(pipe.retriever.rows), "embedder": pipe.info()["embedder"],
        "rerank_model": cfg.rerank_model, "max_len": cfg.max_len, "tune_splits": TUNE_SPLITS, "configs": ALL_CONFIGS,
        "time": time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(meta, open(os.path.join(OUT_DIR, "meta.json"), "w"), ensure_ascii=False, indent=1)
tune_table.to_csv(os.path.join(OUT_DIR, "summary_dev.csv"), encoding="utf-8-sig")
pd.concat([pd.DataFrame(r) for r in tune_rows.values()]).to_csv(os.path.join(OUT_DIR, "per_question_dev.csv"), index=False, encoding="utf-8-sig")
subset_table(tune_rows).to_csv(os.path.join(OUT_DIR, "subsets_dev.csv"), encoding="utf-8-sig")
chg_table.to_csv(os.path.join(OUT_DIR, "changes_dev.csv"), encoding="utf-8-sig")
nf.to_csv(os.path.join(OUT_DIR, "never_found_dev.csv"), index=False, encoding="utf-8-sig")
if final_table is not None:
    final_table.to_csv(os.path.join(OUT_DIR, "summary_final.csv"), encoding="utf-8-sig")
    pd.concat([pd.DataFrame(r) for r in final_rows.values()]).to_csv(os.path.join(OUT_DIR, "per_question_final.csv"), index=False, encoding="utf-8-sig")
step("끝")
log(f"저장: {OUT_DIR}")
log(str(sorted(os.listdir(OUT_DIR))))

## 결과 읽는 법

| 볼 것 | 뜻 |
| --- | --- |
| **Hit@1, Hit@3** | 재정렬의 효과가 가장 잘 보이는 숫자예요. R0보다 얼마나 올랐는지 봐요 |
| **Hit@8** | LLM이 읽는 8개 안에 정답이 있는 비율. 후보 수가 8보다 크면 재정렬로 올라갈 수 있어요 |
| **후보 안에 정답** | 재정렬 전 후보 20개 안에 정답이 있는 비율. 이 값이 재정렬로 올릴 수 있는 한계예요 |
| **재정렬 ms** | 질문 하나당 늘어나는 시간. 답변 생성(수 초)에 비해 얼마나 큰지 봐요 |
| **나빠진 문항** | 재정렬이 맞던 순위를 틀리게 만든 경우. 좋아진 수와 같이 봐요 |

**판단 기준 (예시)**
- Hit@3가 뚜렷이 오르고 시간이 1초 안쪽이면 재정렬을 넣어요.
- "후보 안에 정답"이 낮으면 재정렬보다 검색 단계를 먼저 고쳐요.
- R1과 R2의 정확도가 비슷하면, 모델을 하나 덜 올리는 R1(ColBERT)이 메모리 면에서 유리해요.

**주의**
- 시간은 이 런타임의 GPU 기준이에요. 구성끼리 비교하는 데만 쓰고, 발표용 속도는 서버에서 다시 재요.
- dev는 75문항이라 한 문항이 1.3%p예요. 작은 차이는 우연일 수 있어요.
- 검색 점수가 올라도 최종 답변 정답률이 오르는지는 따로 확인해야 해요.